# Ant policy


<p>there are 2 files policies/experts/Ant.pkl which represents a gaussian distribution w a MLP</p>
<p>and a file iof expert trajectories at expert_data/expert_data_Ant-v4.pkl</p>

In [1]:
import numpy as np
import pickle

with open("/Users/dc/cs224r/hw1/cs224r/policies/experts/Ant.pkl", "rb") as f:
    obj = pickle.load(f)

print('loading Ant.pkl')
print(f"obj, {type(obj)}")
print(obj.keys())

policy_obj = obj["GaussianPolicy"]

# this poolicy is a MPL representing a gaussian, defined with mean and std dev. or variance
policy = policy_obj  # shortcut

# ---------- Extract weights ----------
W0 = policy["hidden"]["FeedforwardNet"]["layer_0"]["AffineLayer"]["W"]
b0 = policy["hidden"]["FeedforwardNet"]["layer_0"]["AffineLayer"]["b"]

W1 = policy["hidden"]["FeedforwardNet"]["layer_2"]["AffineLayer"]["W"]
b1 = policy["hidden"]["FeedforwardNet"]["layer_2"]["AffineLayer"]["b"]

W2 = policy["out"]["AffineLayer"]["W"]
b2 = policy["out"]["AffineLayer"]["b"]

log_std = policy["logstdevs_1_Da"]

# ---------- Extract normalization ----------
stdizer = policy["obsnorm"]["Standardizer"]

mean = stdizer["mean_1_D"]
meansq = stdizer["meansq_1_D"]

var = meansq - mean**2
std = np.sqrt(np.maximum(var, 1e-8))

# ---------- Remove batch dim ----------
mean = mean.squeeze(0)
std = std.squeeze(0)
log_std = log_std.squeeze(0)

# ---------- Expert policy ----------
class AntExpertPolicy:

    def act(self, obs, deterministic=True):

        obs = np.asarray(obs, dtype=np.float32)

        if obs.ndim == 1:
            obs = obs[None, :]

        # normalize
        x = (obs - mean) / (std + 1e-8)

        # layer 1
        x = np.tanh(x @ W0 + b0)

        # layer 2
        x = np.tanh(x @ W1 + b1)

        # output mean
        mu = x @ W2 + b2

        if deterministic:
            a = mu
        else:
            a = mu + np.exp(log_std) * np.random.randn(*mu.shape)

        return a[0]

expert = AntExpertPolicy()

print("Expert reconstructed ✔")

loading Ant.pkl
obj, <class 'dict'>
dict_keys(['GaussianPolicy', 'nonlin_type'])
Expert reconstructed ✔


/var/folders/j2/z3bgs73s7_d7h21sw46sk_4c0000gn/T/ipykernel_97519/3654406774.py:5: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  obj = pickle.load(f)


In [ ]:
# pytorch version can be distributed





